In [1]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


25/12/11 15:04:10 WARN Utils: Your hostname, Manojrams-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.219 instead (on interface en0)
25/12/11 15:04:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-154d5ecd-248e-410a-9316-fa601f90a994;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in c

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [2]:

# Cell 0 — Optional: ensure Spark & paths helpers are available
# (same pattern you use in other notebooks)

import sys
import importlib
import os

# Adjust PROJECT_ROOT if needed
PROJECT_ROOT = "/Users/manojrammopati/Public/Projects/bupa_insurance_project"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Add Silver utils into path
silver_dir = os.path.join(PROJECT_ROOT, "_02_Silver")
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

import utils_silver
importlib.reload(utils_silver)

print("Loaded utils_silver from:", utils_silver.__file__)


✅ utils_silver.py loaded
✅ utils_silver.py loaded
Loaded utils_silver from: /Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver/utils_silver.py


# Cell 1 – Intro markdown (comment)

## ML Scoring – Claims Fraud Risk (Is_Fraudulent_Claim)

Goal: load the best fraud model trained on ft_claims_risk_split and score
a fresh set of claims, writing scored output to Gold (Delta + table).
Steps:
1. Imports & configuration
2. Load ft_claims_risk_split and select rows to score
3. Apply same null-handling as training
4. Load best fraud model from ADLS
5. Score and compute fraud probability
6. Write scored results to Gold + register Spark table


# Cell 2 – Imports & configuration

In [12]:
# Cell 2 — Imports & config

from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

STORAGE_ACCOUNT = "clientdatastorage"
GOLD_CONTAINER  = "golddata"
DB_GOLD         = "bupa_gold"

# Same feature table as training
FT_CLAIMS_RISK_SPLIT_PATH = (
    f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/ft_claims_risk_split"
)

# Model paths (MUST match training notebook)
MODEL_BASE_PATH = (
    f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/models"
)

# 👇 Adjust this to whatever you used in the training notebook printout
MODEL_NAME = "claims_risk_best"

MODEL_PATH = f"{MODEL_BASE_PATH}/{MODEL_NAME}"

# Where we store the scored output
SCORED_BASE_PATH = (
    f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/scored_claims"
)
SCORED_PATH = f"{SCORED_BASE_PATH}/claims_fraud"

ML_MON_PATH = f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/ml_monitoring_view"

print("Feature table      :", FT_CLAIMS_RISK_SPLIT_PATH)
print("Fraud model path   :", MODEL_PATH)
print("Scored output path :", SCORED_PATH)
print("ML monitoring view path :", ML_MON_PATH)


Feature table      : abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_claims_risk_split
Fraud model path   : abfss://golddata@clientdatastorage.dfs.core.windows.net/models/claims_risk_best
Scored output path : abfss://golddata@clientdatastorage.dfs.core.windows.net/scored_claims/claims_fraud
ML monitoring view path : abfss://golddata@clientdatastorage.dfs.core.windows.net/ml_monitoring_view


# Cell 3 – Load feature table & pick rows to score

In [6]:
# Cell 3 — Load ft_claims_risk_split and select scoring population

LABEL_COL = "Is_Fraudulent_Claim"

claims_all = (
    spark.read
         .format("delta")
         .load(FT_CLAIMS_RISK_SPLIT_PATH)
)

print("Total rows in ft_claims_risk_split:", claims_all.count())
claims_all.printSchema()

# For scoring we typically want all dq-valid rows, regardless of dataset_split
claims_scoring = (
    claims_all
        .filter((F.col("dq_money_valid") == 1) & (F.col("dq_date_valid") == 1))
)

print("Rows to score (dq-valid):", claims_scoring.count())
claims_scoring.select("Claim_ID", "dataset_split", LABEL_COL).show(5, truncate=False)


25/12/11 15:20:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Total rows in ft_claims_risk_split: 558211
root
 |-- Claim_ID: string (nullable = true)
 |-- Member_Key: string (nullable = true)
 |-- Provider_ID: string (nullable = true)
 |-- Claim_Type_Name: string (nullable = true)
 |-- Claim_Type_Code: string (nullable = true)
 |-- Claim_Status: string (nullable = true)
 |-- Claim_Amount_GBP: double (nullable = true)
 |-- Payout_GBP: double (nullable = true)
 |-- Payout_to_Amount_Ratio: double (nullable = true)
 |-- Days_To_Settle: integer (nullable = true)
 |-- High_Cost_Claim_Flag: integer (nullable = true)
 |-- Claim_Fraud_Label: integer (nullable = true)
 |-- Provider_Fraud_Label: integer (nullable = true)
 |-- Provider_Risk_Tier: string (nullable = true)
 |-- dq_money_valid: integer (nullable = true)
 |-- dq_date_valid: integer (nullable = true)
 |-- Is_Fraudulent_Claim: integer (nullable = true)
 |-- Is_High_Cost: integer (nullable = true)
 |-- dataset_split: string (nullable = true)



Rows to score (dq-valid): 558211


+---------+-------------+-------------------+
|Claim_ID |dataset_split|Is_Fraudulent_Claim|
+---------+-------------+-------------------+
|CLM110013|train        |0                  |
|CLM110020|train        |0                  |
|CLM110025|train        |0                  |
|CLM110026|train        |1                  |
|CLM110041|train        |0                  |
+---------+-------------+-------------------+
only showing top 5 rows



# Cell 4 – Define features & null-handling (same as training)

In [7]:
# Cell 4 — Define feature columns & null-handling

cols = claims_scoring.columns

numeric_candidates = [
    "Claim_Amount_GBP",
    "Payout_GBP",
    "Payout_to_Amount_Ratio",
    "Days_To_Settle",
]

categorical_candidates = [
    "Claim_Type_Name",
    "Claim_Status",
    "Claim_Type_Code",
    "Provider_Risk_Tier",
    "Provider_ID",
]

numeric_cols = [c for c in numeric_candidates if c in cols]
cat_cols     = [c for c in categorical_candidates if c in cols]

print("Numeric features   :", numeric_cols)
print("Categorical features:", cat_cols)

id_cols = [c for c in ["Claim_ID", "Member_Key", "Provider_ID"] if c in cols]

def prep_nulls(df: DataFrame) -> DataFrame:
    """
    Same simple null-handling as training:
    numeric -> 0.0, categorical -> 'Unknown'.
    """
    d = df
    for c in numeric_cols:
        d = d.withColumn(c, F.when(F.col(c).isNull(), F.lit(0.0)).otherwise(F.col(c)))
    for c in cat_cols:
        d = d.withColumn(c, F.when(F.col(c).isNull(), F.lit("Unknown")).otherwise(F.col(c)))
    return d


Numeric features   : ['Claim_Amount_GBP', 'Payout_GBP', 'Payout_to_Amount_Ratio', 'Days_To_Settle']
Categorical features: ['Claim_Type_Name', 'Claim_Status', 'Claim_Type_Code', 'Provider_Risk_Tier', 'Provider_ID']


# Cell 5 – Load best fraud model & score

In [8]:
# Cell 5 — Load best fraud model & score

print("Loading fraud model from:", MODEL_PATH)
loaded_model = PipelineModel.load(MODEL_PATH)

features_prepped = prep_nulls(claims_scoring)

scored_raw = loaded_model.transform(features_prepped)

# Convert probability vector -> probability of class 1 (fraud)
scored = (
    scored_raw
        .withColumn("prob_arr", vector_to_array("probability"))
        .withColumn("fraud_prob", F.col("prob_arr")[1])
)

# Final tidy projection
keep_cols = (
    id_cols
    + [LABEL_COL]            # will be null in production, but useful in test env
    + [
        "fraud_prob",
        "prediction",
        "Claim_Type_Name",
        "Claim_Status",
        "Claim_Amount_GBP",
        "Payout_GBP",
        "Payout_to_Amount_Ratio",
        "Provider_Risk_Tier",
        "dataset_split",
    ]
)

scored_final = scored.select(*[c for c in keep_cols if c in scored.columns])

print("Scored schema:")
scored_final.printSchema()
scored_final.show(10, truncate=False)


Loading fraud model from: abfss://golddata@clientdatastorage.dfs.core.windows.net/models/claims_risk_best


Scored schema:
root
 |-- Claim_ID: string (nullable = true)
 |-- Member_Key: string (nullable = true)
 |-- Provider_ID: string (nullable = true)
 |-- Is_Fraudulent_Claim: integer (nullable = true)
 |-- fraud_prob: double (nullable = true)
 |-- prediction: double (nullable = false)
 |-- Claim_Type_Name: string (nullable = true)
 |-- Claim_Status: string (nullable = true)
 |-- Claim_Amount_GBP: double (nullable = true)
 |-- Payout_GBP: double (nullable = true)
 |-- Payout_to_Amount_Ratio: double (nullable = true)
 |-- Provider_Risk_Tier: string (nullable = true)
 |-- dataset_split: string (nullable = true)



+---------+----------+-----------+-------------------+-------------------+----------+---------------+------------+------------------+----------+----------------------+------------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Is_Fraudulent_Claim|fraud_prob         |prediction|Claim_Type_Name|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Provider_Risk_Tier|dataset_split|
+---------+----------+-----------+-------------------+-------------------+----------+---------------+------------+------------------+----------+----------------------+------------------+-------------+
|CLM110013|BENE15435 |PRV51790   |0                  |0.30197347924196893|0.0       |Travel         |Settled     |244.07511199775644|200.0     |0.8194198841618822    |Low risk          |train        |
|CLM110020|BENE53030 |PRV53679   |0                  |0.30197347924196893|0.0       |Hospital       |Settled     |4117.6845027538575|2600.0    |0.6314228295686937    |Low risk          |train     

# Cell 6 – Write scored fraud results to Gold + register table

In [10]:
DB_GOLD = "bupa_gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")

DataFrame[]

In [13]:
# Cell 6 — Persist scored fraud claims to Gold & create table in bupa_gold

(
    scored_final
        .write
        .format("delta")
        .mode("overwrite")   # or "append" if you want run history
        .option("overwriteSchema", "true")
        .save(SCORED_PATH)
)

(
    
    scored_final
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")  # or "append" for history
        .save(ML_MON_PATH)
)

print("✅ Wrote scored fraud claims to:", SCORED_PATH, "and to ML monitoring view path:", ML_MON_PATH)

# Register / replace a managed table
TABLE_NAME = "scored_claims_fraud"

spark.sql(f"DROP TABLE IF EXISTS {DB_GOLD}.{TABLE_NAME}")

spark.sql(f"""
    CREATE TABLE {DB_GOLD}.{TABLE_NAME}
    USING DELTA
    LOCATION '{SCORED_PATH}'
""")

print(f"✅ Registered table: {DB_GOLD}.{TABLE_NAME}")
spark.table(f"{DB_GOLD}.{TABLE_NAME}").show(10, truncate=False)


25/12/11 15:48:04 WARN DAGScheduler: Broadcasting large task binary with size 1061.3 KiB
25/12/11 15:48:28 WARN DAGScheduler: Broadcasting large task binary with size 1061.3 KiB


✅ Wrote scored fraud claims to: abfss://golddata@clientdatastorage.dfs.core.windows.net/scored_claims/claims_fraud and to ML monitoring view path: abfss://golddata@clientdatastorage.dfs.core.windows.net/ml_monitoring_view
✅ Registered table: bupa_gold.scored_claims_fraud


+---------+----------+-----------+-------------------+-------------------+----------+---------------+------------+------------------+----------+----------------------+------------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Is_Fraudulent_Claim|fraud_prob         |prediction|Claim_Type_Name|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Provider_Risk_Tier|dataset_split|
+---------+----------+-----------+-------------------+-------------------+----------+---------------+------------+------------------+----------+----------------------+------------------+-------------+
|CLM110013|BENE15435 |PRV51790   |0                  |0.30197347924196893|0.0       |Travel         |Settled     |244.07511199775644|200.0     |0.8194198841618822    |Low risk          |train        |
|CLM110020|BENE53030 |PRV53679   |0                  |0.30197347924196893|0.0       |Hospital       |Settled     |4117.6845027538575|2600.0    |0.6314228295686937    |Low risk          |train     